# 05 — Phân tích lỗi, diễn giải và thí nghiệm bổ sung

Đồ án Máy học — Bank Marketing (Nhóm 14). Chạy theo thứ tự từ trên xuống dưới.

Giai đoạn 8: Phân tích lỗi và diễn giải mô hình (Cẩm nang giai đoạn 8).

- Phân tích lỗi theo nhóm khách hàng (recall của lớp yes + số gọi nhầm).
- Hệ số Logistic Regression (odds ratio) - diễn giải tuyến tính.
- Permutation importance trên test (chỉ để diễn giải, không chọn mô hình).
- Cây nông max_depth=3 để minh họa luật kinh doanh.
- SHAP (nếu cài được): TreeExplainer trên mô hình cuối.

In [ ]:
import json
import sys
from pathlib import Path
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier, plot_tree
sys.path.insert(0, str(Path("..").resolve()))
from src.config import FIGURES_DIR, MODELS_DIR, REPORTS_DIR, SEED
from src.data import load_clean, stratified_split
from src.evaluate import compute_metrics
from src.features import build_pipeline, feature_names_out
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

In [ ]:
df = load_clean()
X_train, X_test, y_train, y_test = stratified_split(df)
final = joblib.load(MODELS_DIR / "model.joblib")
try:
    final.set_params(clf__n_jobs=1)  # tránh song song lồng nhau với permutation_importance
except Exception:
    pass
thr = json.loads((MODELS_DIR / "threshold.json").read_text(encoding="utf-8"))
thr_f1 = thr["threshold_f1"]
p = final.predict_proba(X_test)[:, 1]
pred = (p >= thr_f1).astype(int)
summary = json.loads((REPORTS_DIR / "final_summary.json").read_text(encoding="utf-8"))
out = {"final_model": summary["final_model"], "threshold_f1": thr_f1}

## 8.1 Phân tích lỗi theo nhóm ---

In [ ]:
err = X_test.copy()
err["y"] = y_test.values
err["p"] = p
err["pred"] = pred
err["contacted_before"] = (err["pdays"] != 999).astype(int)
err["is_fp"] = ((err["y"] == 0) & (err["pred"] == 1)).astype(int)
err["is_fn"] = ((err["y"] == 1) & (err["pred"] == 0)).astype(int)
group_tables = {}
for c in ["job", "poutcome", "month", "contacted_before", "contact", "education"]:
    pos = err[err.y == 1]
    tbl = pd.DataFrame({
        "n_yes": pos.groupby(c).size(),
        "recall": pos.groupby(c)["pred"].mean(),
        "n_fp": err.groupby(c)["is_fp"].sum(),
        "n_fn": err.groupby(c)["is_fn"].sum(),
    }).sort_values("recall")
    group_tables[c] = tbl.round(3)
    tbl.to_csv(REPORTS_DIR / f"error_by_{c}.csv")
out["error_by_group"] = {c: t.to_dict("index") for c, t in group_tables.items()}
print("=== Recall của lớp yes theo nhóm (thấp nhất) ===")
for c, t in group_tables.items():
    print(f"-- {c} --")
    print(t.head(4).to_string())

## 8.2 Hệ số Logistic Regression (tuned, fit trên train) ---

In [ ]:
from sklearn.linear_model import LogisticRegression
best_params = json.loads((REPORTS_DIR / "best_params.json").read_text(encoding="utf-8"))
lr_kwargs = {k.replace("clf__", ""): v for k, v in best_params["LogReg"].items()}
lr = build_pipeline(LogisticRegression(max_iter=3000, class_weight="balanced",
                                       random_state=SEED, **lr_kwargs))
lr.fit(X_train, y_train)
names = feature_names_out(lr)
coef = pd.Series(lr.named_steps["clf"].coef_[0], index=names)
odds = np.exp(coef).sort_values(ascending=False)
out["logreg_odds_top10"] = {k: round(float(v), 3) for k, v in odds.head(10).items()}
out["logreg_odds_bottom10"] = {k: round(float(v), 3) for k, v in odds.tail(10).items()}
print("Odds ratio cao nhất:", out["logreg_odds_top10"])
print("Odds ratio thấp nhất:", out["logreg_odds_bottom10"])
fig, ax = plt.subplots(figsize=(9, 5))
pd.concat([odds.head(8), odds.tail(8)]).iloc[::-1].plot(kind="barh", ax=ax, color="#1f77b4")
ax.set_title("Odds ratio - Logistic Regression (top/bottom 8)")
ax.set_xlabel("Odds ratio (exp(coef))")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig11_logreg_odds.png", dpi=150)
plt.close(fig)

## 8.3 Permutation importance trên test ---

In [ ]:
print(">>> Permutation importance (10 repeats, scoring=PR-AUC) ...", flush=True)
r = permutation_importance(final, X_test, y_test, scoring="average_precision",
                           n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.Series(r.importances_mean, index=X_test.columns).sort_values(ascending=False)
imp_std = pd.Series(r.importances_std, index=X_test.columns)
imp_df = pd.DataFrame({"importance": imp, "std": imp_std}).round(4)
imp_df.to_csv(REPORTS_DIR / "permutation_importance.csv")
out["permutation_importance_top10"] = {
    k: round(float(v), 4) for k, v in imp.head(10).items()}
print("Permutation importance top 10:", out["permutation_importance_top10"])
fig, ax = plt.subplots(figsize=(9, 5))
imp.head(12).iloc[::-1].plot(kind="barh", ax=ax, color="#1f77b4")
ax.set_title("Permutation importance (giảm PR-AUC khi hoán vị biến) - test")
ax.set_xlabel("Giảm PR-AUC trung bình")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig12_permutation_importance.png", dpi=150)
plt.close(fig)

## 8.4 Cây nông minh họa luật ---

In [ ]:
shallow = build_pipeline(DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=100, class_weight="balanced", random_state=SEED))
shallow.fit(X_train, y_train)
fig = plt.figure(figsize=(20, 9))
plot_tree(shallow.named_steps["clf"],
          feature_names=feature_names_out(shallow),
          class_names=["no", "yes"], filled=True, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig13_shallow_tree.png", dpi=150)
plt.close(fig)
print("saved fig13_shallow_tree.png")

## 8.5 SHAP (tùy chọn) ---

In [ ]:
try:
    import shap
    print(">>> SHAP TreeExplainer ...", flush=True)
    X_prep = final.named_steps["prep"].transform(
        final.named_steps["fe"].transform(X_test))
    feats = feature_names_out(final)
    sample = X_prep[np.random.default_rng(SEED).choice(len(X_prep), 500, replace=False)]
    explainer = shap.TreeExplainer(final.named_steps["clf"])
    sv = explainer.shap_values(sample)
    vals = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)
    mean_abs = np.abs(vals).mean(axis=0)
    top = np.argsort(mean_abs)[::-1][:12]
    out["shap_top12"] = {str(feats[i]): round(float(mean_abs[i]), 4) for i in top}
    fig, ax = plt.subplots(figsize=(9, 5))
    pd.Series({str(feats[i]): mean_abs[i] for i in top[::-1]}).plot(
        kind="barh", ax=ax, color="#1f77b4")
    ax.set_title("SHAP - |SHAP value| trung bình (top 12, mẫu 500 dòng test)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig14_shap_summary.png", dpi=150)
    plt.close(fig)
    print("SHAP top 12:", out["shap_top12"])
except Exception as e:
    out["shap_status"] = f"không chạy được ({type(e).__name__}: {e}); permutation importance đã đủ theo cẩm nang."
    print("SHAP bỏ qua:", e)

(REPORTS_DIR / "analysis_summary.json").write_text(
    json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu reports/analysis_summary.json")

Giai đoạn 9: Thí nghiệm bổ sung (Cẩm nang giai đoạn 9).

Làm 5 thí nghiệm (ưu tiên 9.1, 9.2, 9.5 + 9.3, 9.4):
- 9.1 duration A/B (RQ4): cùng mô hình đã tuning, có/không duration.
- 9.2 ablation theo nhóm biến (RQ3): 7 tổ hợp với LogReg balanced.
- 9.3 chia theo thời gian (concept drift): mô hình cuối + LogReg, cả bản
  đầy đủ và bản BỎ nhóm biến vĩ mô (kiểm tra giả thuyết "đồng hồ thời gian").
- 9.4 hiệu chỉnh xác suất (calibration): isotonic, Brier trước/sau.
- 9.5 gain/lift + chi phí-lợi ích (RQ5): decile, top-20% capture, độ nhạy
  với các cặp (C, R).
- 9.6 SMOTE so với class_weight (chỉ lấy mẫu lại trong fold train).

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
sys.path.insert(0, str(Path("..").resolve()))
from src.config import FIGURES_DIR, N_SPLITS, REPORTS_DIR, SEED
from src.data import load_clean, stratified_split, time_split
from src.evaluate import cv_report, gain_lift_table
from src.features import (GROUP_CAMPAIGN, GROUP_MACRO, GROUP_PERSONAL,
                          add_features, build_pipeline)
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})
CLASSES = {"LogReg": LogisticRegression, "RF": RandomForestClassifier}
DEFAULT_KWARGS = {"LogReg": {"max_iter": 3000, "class_weight": "balanced", "random_state": SEED},
                  "RF": {"class_weight": "balanced_subsample", "n_jobs": -1, "random_state": SEED}}
def estimator_from_best(name, best_params):
    kwargs = {k.replace("clf__", ""): v for k, v in best_params[name].items()}
    return CLASSES[name](**{**DEFAULT_KWARGS[name], **kwargs})

In [ ]:
df = load_clean()
X_train, X_test, y_train, y_test = stratified_split(df)
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
best_params = json.loads((REPORTS_DIR / "best_params.json").read_text(encoding="utf-8"))
final_summary = json.loads((REPORTS_DIR / "final_summary.json").read_text(encoding="utf-8"))
final_name = final_summary["final_model"]
out = {}

## 9.1 duration A/B (dùng RF đã tuning) ----------

In [ ]:
rf = estimator_from_best("RF", best_params)
model_a = build_pipeline(RandomForestClassifier(**rf.get_params()))
model_b = build_pipeline(RandomForestClassifier(**rf.get_params()), include_duration=True)
ab = cv_report({"A_khong_duration": model_a, "B_co_duration": model_b},
               X_train, y_train, cv)
ab.to_csv(REPORTS_DIR / "duration_ab_cv.csv")
out["duration_ab_cv"] = ab[["roc_auc_cv", "pr_auc_cv"]].round(4).to_dict("index")
print("9.1 duration A/B (CV):", out["duration_ab_cv"])

## 9.2 ablation theo nhóm biến ----------

In [ ]:
groups = {"Personal": GROUP_PERSONAL, "Campaign": GROUP_CAMPAIGN, "Macro": GROUP_MACRO}
combos = {"Personal": ["Personal"], "Campaign": ["Campaign"], "Macro": ["Macro"],
          "Personal+Campaign": ["Personal", "Campaign"],
          "Campaign+Macro": ["Campaign", "Macro"],
          "Personal+Macro": ["Personal", "Macro"],
          "All": ["Personal", "Campaign", "Macro"]}

def merge(names):
    num = sum([groups[n]["num"] for n in names], [])
    cat = sum([groups[n]["cat"] for n in names], [])
    return num, cat

abl = {}
for label, names in combos.items():
    num, cat = merge(names)
    abl[label] = build_pipeline(
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
        num=num, cat=cat)
abl_res = cv_report(abl, X_train, y_train, cv)
abl_res.to_csv(REPORTS_DIR / "ablation_cv.csv")
out["ablation_cv"] = abl_res[["roc_auc_cv", "pr_auc_cv", "recall_cv", "f1_cv"]].round(4).to_dict("index")
print("9.2 ablation (CV):")
print(abl_res[["roc_auc_cv", "pr_auc_cv"]].to_string())

## 9.3 chia theo thời gian ----------

In [ ]:
Xt, Xv, yt, yv = time_split(df)
final_est = estimator_from_best(final_name, best_params)
m_full = build_pipeline(type(final_est)(**final_est.get_params())).fit(Xt, yt)
pv = m_full.predict_proba(Xv)[:, 1]
m_lr = build_pipeline(LogisticRegression(
    max_iter=3000, class_weight="balanced", random_state=SEED)).fit(Xt, yt)
pv_lr = m_lr.predict_proba(Xv)[:, 1]
# Bản bỏ nhóm vĩ mô (kiểm tra giả thuyết "đồng hồ thời gian")
num_nomacro = [c for c in ["age", "campaign", "previous", "pdays_clean", "contacted_before"]]
cat_nomacro = ["job", "marital", "education", "default", "housing", "loan",
               "contact", "month", "day_of_week", "poutcome"]
m_nomacro = build_pipeline(type(final_est)(**final_est.get_params()),
                           num=num_nomacro, cat=cat_nomacro).fit(Xt, yt)
pv_nm = m_nomacro.predict_proba(Xv)[:, 1]
out["time_split"] = {
    "val_yes_rate": round(float(yv.mean()), 4),
    final_name + "_full": {"roc_auc": round(float(roc_auc_score(yv, pv)), 4),
                           "pr_auc": round(float(average_precision_score(yv, pv)), 4)},
    "LogReg_full": {"roc_auc": round(float(roc_auc_score(yv, pv_lr)), 4),
                    "pr_auc": round(float(average_precision_score(yv, pv_lr)), 4)},
    final_name + "_no_macro": {"roc_auc": round(float(roc_auc_score(yv, pv_nm)), 4),
                               "pr_auc": round(float(average_precision_score(yv, pv_nm)), 4)},
}
print("9.3 time split:", out["time_split"])

## 9.4 calibration ----------

In [ ]:
import joblib
from src.config import MODELS_DIR
final = joblib.load(MODELS_DIR / "model.joblib")
p_test = final.predict_proba(X_test)[:, 1]
from sklearn.base import clone as _clone
_cal_base = _clone(final)
try:
    _cal_base.set_params(clf__n_jobs=1)  # tránh song song lồng nhau với CalibratedClassifierCV
except Exception:
    pass
cal = CalibratedClassifierCV(_cal_base, method="isotonic", cv=5).fit(X_train, y_train)
p_cal = cal.predict_proba(X_test)[:, 1]
brier_before = float(brier_score_loss(y_test, p_test))
brier_after = float(brier_score_loss(y_test, p_cal))
out["calibration"] = {"brier_before": round(brier_before, 4),
                      "brier_after": round(brier_after, 4)}
fig, ax = plt.subplots(figsize=(6, 5))
for prob, lab in [(p_test, "Mô hình gốc"), (p_cal, "Sau calibration (isotonic)")]:
    fp, mp = calibration_curve(y_test, prob, n_bins=10, strategy="quantile")
    ax.plot(mp, fp, marker="o", label=lab)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("Xác suất dự đoán (trung bình bin)")
ax.set_ylabel("Tỷ lệ yes thực")
ax.set_title("Calibration curve (test)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig15_calibration_curve.png", dpi=150)
plt.close(fig)
print("9.4 calibration: Brier", brier_before, "->", brier_after)

## 9.5 gain / lift / chi phí-lợi ích ----------

In [ ]:
res, dec = gain_lift_table(y_test, p_test)
dec.to_csv(REPORTS_DIR / "decile_table.csv")
out["decile_table"] = dec.round(4).to_dict("index")
top20 = res[res["called_pct"] <= 0.20]["captured"].iloc[-1]
out["top20_capture"] = round(float(top20), 4)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(res["called_pct"], res["captured"], label="Mô hình cuối")
ax.plot([0, 1], [0, 1], "--", label="Ngẫu nhiên")
ax.set_xlabel("% khách được gọi (xếp theo điểm)")
ax.set_ylabel("% khách đồng ý bắt được")
ax.set_title("Gain chart (test)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig16_gain_chart.png", dpi=150)
plt.close(fig)
# Chi phí-lợi ích: C = chi phí 1 cuộc gọi, R = lợi nhuận khi 1 khách đồng ý.
# GIẢ ĐỊNH của nhóm (không phải số thật của ngân hàng), thử độ nhạy.
cb = {}
for (C, R) in [(1.0, 15.0), (1.0, 10.0), (1.0, 25.0), (2.0, 15.0), (0.5, 15.0)]:
    profit = R * res["y"].cumsum() - C * (res.index + 1)
    k = int(profit.idxmax())
    cb[f"C={C},R={R}"] = {"top_pct": round(float(res.loc[k, "called_pct"] * 100), 1),
                          "capture_pct": round(float(res.loc[k, "captured"] * 100), 1),
                          "profit_max": round(float(profit.loc[k]), 1),
                          "profit_call_all": round(float(profit.iloc[-1]), 1)}
out["cost_benefit"] = cb
out["top20_capture_note"] = "Gọi top 20% theo điểm mô hình bắt được {:.1f}% số người đồng ý.".format(top20 * 100)
print("9.5 top-20% capture:", out["top20_capture"], "| cost-benefit:", cb)

## 9.6 SMOTE so với class_weight ----------

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    prep = ColumnTransformer([
        ("num", StandardScaler(),
         ["age", "campaign", "previous", "pdays_clean", "contacted_before",
          "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["job", "marital", "education", "default", "housing", "loan",
          "contact", "month", "day_of_week", "poutcome"]),
    ])
    smote_pipe = ImbPipeline([
        ("fe", FunctionTransformer(add_features)),
        ("prep", prep),
        ("smote", SMOTE(random_state=SEED)),
        ("clf", LogisticRegression(max_iter=3000, random_state=SEED)),
    ])
    cw_pipe = build_pipeline(LogisticRegression(max_iter=3000, class_weight="balanced",
                                                random_state=SEED))
    sm = cv_report({"LogReg_SMOTE": smote_pipe, "LogReg_class_weight": cw_pipe},
                   X_train, y_train, cv)
    sm.to_csv(REPORTS_DIR / "smote_comparison_cv.csv")
    out["smote_vs_class_weight"] = sm[["roc_auc_cv", "pr_auc_cv", "f1_cv", "recall_cv"]].round(4).to_dict("index")
    print("9.6 SMOTE vs class_weight:", out["smote_vs_class_weight"])
except Exception as e:
    out["smote_status"] = f"không chạy được ({type(e).__name__}: {e})"
    print("9.6 SMOTE bỏ qua:", e)

(REPORTS_DIR / "extra_experiments.json").write_text(
    json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu reports/extra_experiments.json")